In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/crisisMMD/
import sys
sys.path.insert(0,"/content/drive/MyDrive/crisisMMD/")

/content/drive/MyDrive/crisisMMD


In [3]:
import numpy as np
np.random.seed(1337)  # for reproducibility

import os
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils.np_utils import to_categorical
import sys
from sklearn import preprocessing
import pandas as pd
import re
from gensim.models import Word2Vec
from gensim.models import KeyedVectors
import nltk
import math
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
import random
random.seed(1337)
import aidrtokenize as aidrtokenize

def file_exist(file_name):
    if os.path.exists(file_name):
        return True
    else:
        return False

def read_stop_words(file_name):
    if(not file_exist(file_name)):
        print("Please check the file for stop words, it is not in provided location "+file_name)
        sys.exit(0)
    stop_words =[]
    with open(file_name, 'rU') as f:
        for line in f:
            line = line.strip()
            if (line == ""):
                continue
            stop_words.append(line)
    return stop_words;

stop_words_file="bin/stop_words_english.txt"
stop_words = read_stop_words(stop_words_file)

 

def read_train_data(dataFile, MAX_NB_WORDS, MAX_SEQUENCE_LENGTH, delim):
    """
    Prepare the data
    """
    data=[]
    lab=[]
    with open(dataFile, 'rU') as f:
        next(f)    
        for line in f:
            line = line.strip()   
            if (line==""):
                continue                                    
            row=line.split(delim)
            #print(line)
            txt = row[3].strip()
            txt = txt.replace("'", "")
            txt = aidrtokenize.tokenize(txt)

            label = row[6]
            #print(label)
            txt = txt.replace("'", "")
            w_list=[]
            for w in txt.split():
                if w not in stop_words:
                    try:
                        #w=str(w.encode('ascii'))
                        w_list.append(w.encode('utf-8'))
                    except Exception as e:
                        print(w)
                        pass
            #print(w_list)            
            text = b" ".join(w_list)
            text=text.decode('UTF-8')
            #print(type(text))
            # if(len(text)<1):
            #     print txt
            #     continue
            #txt=aidrtokenize.tokenize(txt)
            #txt=[w for w in txt if w not in stop_words]              
            if(isinstance(text, str)):
                data.append(text)
                lab.append(label) 
    print(data)                   
    data_shuf = []
    lab_shuf = []
    index_shuf = [i for i in range(len(data))]
    random.shuffle(index_shuf)
    for i in index_shuf:
        data_shuf.append(data[i])
        lab_shuf.append(lab[i])

    #print(lab_shuf)
    le = preprocessing.LabelEncoder()
    yL=le.fit_transform(lab_shuf)
    #print(yL)
    labels=list(le.classes_)
    label=yL.tolist()
    yC=len(set(label))
    yR=len(label)
    y = np.zeros((yR, yC))
    #print(y,y.dtype)
    #print(yR,type(yR))
    #print(yL,type(yL))
    y[np.arange(yR), yL] = 1
    y=np.array(y,dtype=np.int32)
    
    #print(y)
    # finally, vectorize the text samples into a 2D integer tensor
    tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token="OOV_TOK")
    tokenizer.fit_on_texts(data_shuf )
    sequences = tokenizer.texts_to_sequences(data_shuf)
    
    word_index = tokenizer.word_index
    print('Found %s unique tokens.' % len(word_index))
    
    data = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH)
    
    #labels = to_categorical(np.asarray(labels))
    print('Shape of data tensor:', data.shape)
    #print('Shape of label tensor:', labels.shape)    
    #return data,labels,word_index,dim;        
    return data,y,le,labels,word_index,tokenizer

    
def read_dev_data(dataFile, tokenizer, MAX_SEQUENCE_LENGTH, delim, train_le):
    """
    Prepare the data
    """      
    data=[]
    lab=[]
    with open(dataFile, 'rU') as f:
         next(f)    
         for line in f:
             line = line.strip()   
             if (line==""):
                continue                                    
             row=line.split(delim)
             #print(line)
             txt = row[3].strip()
             txt = txt.replace("'", "")
             txt = aidrtokenize.tokenize(txt)

             label = row[6]
             #print(label)
             txt = txt.replace("'", "")
             #print(txt)
             w_list=[]
             for w in txt.split():
                 if w not in stop_words:
                    try:
                        #w=str(w.encode('ascii'))
                        w_list.append(w.encode('utf-8'))
                    except Exception as e:
                        print(w)
                        pass
             #print(w_list)            
             text = b" ".join(w_list)
             text=text.decode("UTF-8")
             #print(text)
             # if(len(text)<1):
             #     print txt
             #     continue
             #txt=aidrtokenize.tokenize(txt)
             #txt=[w for w in txt if w not in stop_words]              
             if(isinstance(text, str)):
                data.append(text)
                lab.append(label)
    le = train_le
    yL=le.transform(lab)
    labels=list(le.classes_)
    label=yL.tolist()
    yC=len(set(label))
    yR=len(label)
    y = np.zeros((yR, yC))
    print(y,y.dtype)
    print(yR,type(yR))
    print(yL,type(yL))
    x=np.arange(yR)
    print(x,type(x))
    y[x, yL] = 1
    y=np.array(y,dtype=np.int32)
    sequences = tokenizer.texts_to_sequences(data)   
    word_index = tokenizer.word_index
    print('Found %s unique tokens.' % len(word_index))    
    data = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH)
    print('Shape of data tensor:', data.shape)
    return data,y,le,labels,word_index


def read_data_classifier(dataFile, tokenizer, MAX_SEQUENCE_LENGTH, delim, train_le):
    """
    Prepare the data
    """
    data = []
    lab = []
    with open(dataFile, 'rU') as f:
        next(f)
        for line in f:
            line = line.strip()
            if (line == ""):
                continue
            row = line.split(delim)
            txt = row[3].strip()
            txt = txt.replace("'", "")
            txt = aidrtokenize.tokenize(txt)

            label = row[6]

            txt = txt.replace("'", "")
            w_list = []
            for w in txt.split():
                if w not in stop_words:
                    try:
                        # w=str(w.encode('ascii'))
                        w_list.append(w.encode('utf-8'))
                    except Exception as e:
                        # print(w)
                        # print(e)
                        pass
            
            text = b" ".join(w_list)

            # if(len(text)<1):
            #     print txt
            #     continue
            # txt=aidrtokenize.tokenize(txt)
            # txt=[w for w in txt if w not in stop_words]
            if (isinstance(text, bytes)):
                data.append(text)
                lab.append(label)
            else:
                print("not text: " + text)

    # le = train_le  # preprocessing.LabelEncoder()
    # yL = le.transform(lab)
    # labels = list(le.classes_)
    #
    # label = yL.tolist()
    # yC = len(set(label))
    # yR = len(label)
    # y = np.zeros((yR, yC))
    # y[np.arange(yR), yL] = 1
    # y = np.array(y, dtype=np.int32)

    sequences = tokenizer.texts_to_sequences(data)
    word_index = tokenizer.word_index
    print('Found %s unique tokens.' % len(word_index))
    data_x = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH)
    print('Shape of data tensor:', data_x.shape)
    return data_x, data, lab, #le, labels, word_index

def load_embedding(fileName):
    print('Indexing word vectors.')    
    embeddings_index = {}    
    f = open(fileName)
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
    f.close()    
    print('Found %s word vectors.' % len(embeddings_index))
    return embeddings_index;

def prepare_embedding(word_index, model, MAX_NB_WORDS, EMBEDDING_DIM):
    
    # prepare embedding matrix
    nb_words = min(MAX_NB_WORDS, len(word_index)+1)    
    embedding_matrix = np.zeros((nb_words, EMBEDDING_DIM),dtype=np.float32)
    print(len(embedding_matrix))
    for word, i in word_index.items():
        if i >= nb_words:
            continue
        try:
            embedding_vector = model[word][0:EMBEDDING_DIM] #embeddings_index.get(word)
            embedding_matrix[i] = np.asarray(embedding_vector,dtype=np.float32)
        except KeyError:
            try:
                print(word +" not found... assigning zeros")
                rng = np.random.RandomState()           
                #embedding_vector = rng.randn(EMBEDDING_DIM) #np.random.random(num_features)
                embedding_vector = np.zeros(EMBEDDING_DIM)  # np.random.random(num_features)
                embedding_matrix[i] = np.asarray(embedding_vector,dtype=np.float32)
            except KeyError:    
                continue      
    return embedding_matrix;

def str_to_indexes(s):
    alphabet = "abcdefghijklmnopqrstuvwxyz0123456789-,;.!?:'\"/\\|_@#$%^&*~`+-=<>()[]{}"
    input_size = 1014
    length = input_size
    alphabet_size = len(alphabet)
    char_dict = {}  # Maps each character to an integer
    # self.no_of_classes = num_of_classes
    for idx, char in enumerate(alphabet):
        char_dict[char] = idx + 1
    length = input_size


    """
    Convert a string to character indexes based on character dictionary.
    Args:
        s (str): String to be converted to indexes
    Returns:
        str2idx (np.ndarray): Indexes of characters in s
    """
    s = s.lower()
    max_length = min(len(s), length)
    str2idx = np.zeros(length, dtype='int64')
    for i in range(1, max_length + 1):
        c = s[-i]
        if c in char_dict:
            str2idx[i - 1] = char_dict[c]
    return str2idx

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:34: DeprecationWarning: 'U' mode is deprecated


In [12]:
import numpy as np
# for reproducibility
seed = 1337
np.random.seed(seed)

#from __future__ import print_function
import os
import numpy as np
np.random.seed(1337)

from keras.layers import Conv1D, MaxPooling1D, Embedding
import shlex
from subprocess import Popen, PIPE
from collections import Counter
import random
from keras.layers import concatenate
from keras.constraints import max_norm
from keras.layers import Input, Dense, Embedding, Conv2D, MaxPool2D, MaxPooling2D
from keras.layers import Reshape, Flatten, Dropout, Concatenate

def get_exitcode_stdout_stderr(cmd):
    """
    Execute the external command and get its exitcode, stdout and stderr.
    """
    args = shlex.split(cmd)

    proc = Popen(args, stdout=PIPE, stderr=PIPE)
    out, err = proc.communicate()
    exitcode = proc.returncode
    #
    return exitcode, out, err
    
def label_one_hot(yL):
    label=yL.tolist()
    yC=len(set(label))
    yR=len(label)
    y = np.zeros((yR, yC))
    y[np.arange(yR), yL] = 1
    y=np.array(y,dtype=np.int32)
    return y  
    
def upsampling(train_x,train_y):
    ########## Upsampling    
    y_true=np.argmax(train_y, axis = 1)
    smote = ""#SMOTE(ratio=0.5, kind='borderline1',n_jobs=5)
    X_resampled, y_resampled = smote.fit_sample(train_x,y_true)
  
    ########## Shuffling  
    combined = list(zip(X_resampled, y_resampled))
    random.shuffle(combined)
    X_resampled[:], y_resampled[:] = zip(*combined)
    y_resampled_true=label_one_hot(y_resampled)
    dimension = X_resampled.shape[1]
    y_resampled_true=label_one_hot(y_resampled)
    print(len(X_resampled))
    X_resampled=np.array(X_resampled)
    print(X_resampled.shape)    
    counts = Counter(y_resampled)
    print(counts)   
    return X_resampled, y_resampled_true, dimension
  
def text_cnn(embedding_matrix,word_index,MAX_NB_WORDS,EMBEDDING_DIM,MAX_SEQUENCE_LENGTH,inputs):
    nb_words = min(MAX_NB_WORDS, len(word_index)+1)
    embedding_layer=Embedding(output_dim=EMBEDDING_DIM, input_dim=nb_words, weights=[embedding_matrix], input_length=MAX_SEQUENCE_LENGTH,trainable=True)(inputs)
    # embedding_layer=Embedding(output_dim=EMBEDDING_DIM, input_dim=nb_words, input_length=MAX_SEQUENCE_LENGTH,trainable=False)(inputs)

    ########## CNN: Filtering with Max pooling:
    branches = [] # models to be merged
    filter_window_sizes=[2,3,4,5]
    pool_size=2
    num_filters=[100,150,200,300]
    for filter_len,nb_filter in zip(filter_window_sizes,num_filters):
        branch = embedding_layer
        branch=Conv1D(filters=nb_filter,
                                 kernel_size=int(filter_len),
                                 padding='valid',
                                 activation='relu',
                                 strides=1,
                                 kernel_initializer='glorot_uniform',
                                 kernel_constraint=max_norm(3), bias_constraint=max_norm(3))(branch)
        branch=MaxPooling1D(pool_size=pool_size)(branch)
        branch=Flatten()(branch)
        branches.append(branch)
    merged_model=concatenate(branches)

    return merged_model


def text_cnn_2d(embedding_matrix,word_index,MAX_NB_WORDS,EMBEDDING_DIM,MAX_SEQUENCE_LENGTH,inputs):
    # this returns a tensor
    print("Creating Model...")
    # inputs = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype='int32')
    # embedding = Embedding(input_dim=vocabulary_size, output_dim=embedding_dim, input_length=sequence_length)(inputs)

    nb_words = min(MAX_NB_WORDS, len(word_index)+1)
    embedding=Embedding(output_dim=EMBEDDING_DIM, input_dim=nb_words, weights=[embedding_matrix], input_length=MAX_SEQUENCE_LENGTH,trainable=True)(inputs)

    reshape = Reshape((MAX_SEQUENCE_LENGTH,EMBEDDING_DIM,1))(embedding)
    filter_window_sizes = [2, 3, 4]
    # filter_sizes = [3, 4, 5]
    num_filters = 512
    conv_0 = Conv2D(num_filters, kernel_size=(filter_window_sizes[0], EMBEDDING_DIM), padding='valid', kernel_initializer='normal', activation='relu')(reshape)
    conv_1 = Conv2D(num_filters, kernel_size=(filter_window_sizes[1], EMBEDDING_DIM), padding='valid', kernel_initializer='normal', activation='relu')(reshape)
    conv_2 = Conv2D(num_filters, kernel_size=(filter_window_sizes[2], EMBEDDING_DIM), padding='valid', kernel_initializer='normal', activation='relu')(reshape)

    maxpool_0 = MaxPool2D(pool_size=(MAX_SEQUENCE_LENGTH - filter_window_sizes[0] + 1, 1), strides=(1,1), padding='valid')(conv_0)
    maxpool_1 = MaxPool2D(pool_size=(MAX_SEQUENCE_LENGTH - filter_window_sizes[1] + 1, 1), strides=(1,1), padding='valid')(conv_1)
    maxpool_2 = MaxPool2D(pool_size=(MAX_SEQUENCE_LENGTH - filter_window_sizes[2] + 1, 1), strides=(1,1), padding='valid')(conv_2)

    concatenated_tensor = Concatenate(axis=1)([maxpool_0, maxpool_1, maxpool_2])
    merged_model = Flatten()(concatenated_tensor)
    return merged_model



## sentence CNN by Y.Kim
def kimCNN(embedding_matrix,word_index,MAX_NB_WORDS,EMBEDDING_DIM,MAX_SEQUENCE_LENGTH,sequence_input):
    """
    Convolution neural network model for sentence classification.
    Parameters
    ----------
    EMBEDDING_DIM: Dimension of the embedding space.
    MAX_SEQUENCE_LENGTH: Maximum length of the sentence.
    MAX_NB_WORDS: Maximum number of words in the vocabulary.
    embeddings_index: A dict containing words and their embeddings.
    word_index: A dict containing words and their indices.
    labels_index: A dict containing the labels and their indices.
    Returns
    -------
    compiled keras model
    """
    print('Preparing embedding matrix.')
    nb_words = min(MAX_NB_WORDS, len(word_index)+1)
    embedding_layer = Embedding(output_dim=EMBEDDING_DIM, input_dim=nb_words, weights=[embedding_matrix], input_length=MAX_SEQUENCE_LENGTH,trainable=True)

    embedded_sequences = embedding_layer(sequence_input)
    print(embedded_sequences.shape)
    # add first conv filter
    embedded_sequences = Reshape((MAX_SEQUENCE_LENGTH, EMBEDDING_DIM, 1))(embedded_sequences)
    x = Conv2D(300, (5, EMBEDDING_DIM), activation='relu')(embedded_sequences)
    x = MaxPool2D((MAX_SEQUENCE_LENGTH - 5 + 1, 1))(x)

    # add second conv filter.
    y = Conv2D(300, (4, EMBEDDING_DIM), activation='relu')(embedded_sequences)
    y = MaxPool2D((MAX_SEQUENCE_LENGTH - 4 + 1, 1))(y)


    # add third conv filter.
    z = Conv2D(300, (3, EMBEDDING_DIM), activation='relu')(embedded_sequences)
    z = MaxPool2D((MAX_SEQUENCE_LENGTH - 3 + 1, 1))(z)

    # add third conv filter.
    z1 = Conv2D(300, (2, EMBEDDING_DIM), activation='relu')(embedded_sequences)
    z1 = MaxPool2D((MAX_SEQUENCE_LENGTH - 2 + 1, 1))(z1)

    # add third conv filter.
    w1 = Conv2D(300, (1, EMBEDDING_DIM), activation='relu')(embedded_sequences)
    w1 = MaxPool2D((MAX_SEQUENCE_LENGTH - 1 + 1, 1))(w1)

    alpha = concatenate([w1,z1])

    # flatted the pooled features.
    merged_model = Flatten()(alpha)

    return merged_model

In [18]:
!pip install cPickle

ERROR: Could not find a version that satisfies the requirement cPickle (from versions: none)
ERROR: No matching distribution found for cPickle


In [19]:
import numpy as np
from sklearn import metrics
import sys
import os
import sklearn.metrics as metrics
from sklearn import preprocessing
import pandas as pd
import re
import pandas as pd
from sklearn.metrics import roc_auc_score

def roc_auc_score_multiclass(actual_class, pred_class, average = "weighted"):

  #creating a set of all the unique classes using the actual class list
    unique_class = set(actual_class)
    roc_auc_dict = {}
    for per_class in unique_class:
    #creating a list of all the classes except the current class
        other_class = [x for x in unique_class if x != per_class]

        #marking the current class as 1 and all other classes as 0
        new_actual_class = [0 if x in other_class else 1 for x in actual_class]
        new_pred_class = [0 if x in other_class else 1 for x in pred_class]

        #using the sklearn metrics method to calculate the roc_auc_score
        roc_auc = roc_auc_score(new_actual_class, new_pred_class, average = average)
        roc_auc_dict[per_class] = roc_auc

    list_values = [v for v in roc_auc_dict.values()]
    average = np.average(list_values)
    return average


def performance_measure(y_true,y_pred,le):

    acc=P=R=F1=AUC=0.0
    report=""
    AUC = roc_auc_score_multiclass(y_true, y_pred)

    #print(roc_auc_multiclass)
    try:
       acc=metrics.accuracy_score(y_true,y_pred)
       P=metrics.precision_score(y_true,y_pred,average="weighted")
       R=metrics.recall_score(y_true,y_pred,average="weighted")
       F1=metrics.f1_score(y_true,y_pred,average="weighted")
       report=metrics.classification_report(y_true, y_pred)
    except Exception as e:
        print (e)
        pass
    return AUC,acc,P,R,F1,report


def performance_measure_cnn(y_true, y_prob, le):
    y_true = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_prob, axis=1)


    y_true = le.inverse_transform(y_true)
    y_pred = le.inverse_transform(y_pred)
    acc = P = R = F1 = AUC = 0.0
    report = ""
    AUC = roc_auc_score_multiclass(y_true, y_pred)
    try:
        acc = metrics.accuracy_score(y_true, y_pred)
        P = metrics.precision_score(y_true, y_pred, average="weighted")
        R = metrics.recall_score(y_true, y_pred, average="weighted")
        F1 = metrics.f1_score(y_true, y_pred, average="weighted")
        report = metrics.classification_report(y_true, y_pred)

    except Exception as e:
        print (e)
        pass

    return AUC,acc, P, R, F1, report

def performance_measure_classifier(y_true, y_prob, le):
    # y_true = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_prob, axis=1)


    # y_true = le.inverse_transform(y_true)
    y_pred = le.inverse_transform(y_pred)
    acc = P = R = F1 = AUC = 0.0
    report = ""
    AUC = roc_auc_score_multiclass(y_true, y_pred)
    try:
        acc = metrics.accuracy_score(y_true, y_pred)
        P = metrics.precision_score(y_true, y_pred, average="weighted")
        R = metrics.recall_score(y_true, y_pred, average="weighted")
        F1 = metrics.f1_score(y_true, y_pred, average="weighted")
        report = metrics.classification_report(y_true, y_pred)

    except Exception as e:
        print (e)
        pass

    return AUC,acc, P, R, F1, report

def format_conf_mat(y_true,y_pred,le):
    y_true = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_pred, axis=1)


    y_true = le.inverse_transform(y_true)
    y_pred = le.inverse_transform(y_pred)


    conf_mat = pd.crosstab(np.array(y_true), np.array(y_pred), rownames=['gold'], colnames=['pred'], margins=True)
    pred_columns = conf_mat.columns.tolist()
    gold_rows = conf_mat.index.tolist()
    conf_mat_str = ""
    header = "Pred\nGold"
    for h in pred_columns:
        header = header + "\t" + str(h)
    conf_mat_str = header + "\n"
    index = 0
    for r_index, row in conf_mat.iterrows():
        row_str = str(gold_rows[index])  # for class label (name)
        index += 1
        for col_item in row:
            row_str = row_str + "\t" + str(col_item)
        conf_mat_str = conf_mat_str + row_str + "\n"
    return conf_mat_str



def format_conf_mat_classifier(y_true,y_pred,le):
    # y_true = np.argmax(y_true, axis=1)
    y_pred = np.argmax(y_pred, axis=1)


    # y_true = le.inverse_transform(y_true)
    y_pred = le.inverse_transform(y_pred)

    conf_mat = pd.crosstab(np.array(y_true), np.array(y_pred), rownames=['gold'], colnames=['pred'], margins=True)
    pred_columns = conf_mat.columns.tolist()
    gold_rows = conf_mat.index.tolist()
    conf_mat_str = ""
    header = "Pred\nGold"
    for h in pred_columns:
        header = header + "\t" + str(h)
    conf_mat_str = header + "\n"
    index = 0
    for r_index, row in conf_mat.iterrows():
        row_str = str(gold_rows[index])  # for class label (name)
        index += 1
        for col_item in row:
            row_str = row_str + "\t" + str(col_item)
        conf_mat_str = conf_mat_str + row_str + "\n"
    return conf_mat_str

In [27]:
!pip install scikit-learn==0.24.2

     |████████████████████████████████| 22.3 MB 18.1 MB/s 
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.0.2
    Uninstalling scikit-learn-1.0.2:
      Successfully uninstalled scikit-learn-1.0.2


In [ ]:
import os

import tensorflow as tf 
import numpy as np
import sys
from gensim.models import KeyedVectors
from keras.layers import Dense, Input, Dropout, Activation, Flatten, BatchNormalization
from keras.models import Sequential, Model
import keras.callbacks as callbacks
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, LearningRateScheduler, CSVLogger, TensorBoard
import keras
#import performance as performance
import os, errno
import warnings
import datetime
import optparse
import pickle
from time import time
from datetime import datetime
import re
from keras.models import load_model
from sklearn.utils import compute_class_weight

seed = 1337
np.random.seed(seed)

def save_model(model, model_dir, model_file_name, tokenizer, label_encoder):
    if not os.path.exists(model_dir):
        os.makedirs(model_dir)
    base_name = os.path.basename(model_file_name)
    base_name = os.path.splitext(base_name)[0]
    timestr = datetime.now().strftime("%d-%m-%Y_%I-%M-%S")

    model_file = model_dir + "/" + base_name +"_"+timestr+ ".hdf5"
    tokenizer_file = model_dir + "/" + base_name +"_"+timestr+ ".tokenizer"
    label_encoder_file = model_dir + "/" + base_name +"_"+timestr+ ".label_encoder"

    configfile = model_dir + "/" + base_name + ".config"
    configFile = open(configfile, "w")
    configFile.write("model_file=" + model_file + "\n")
    configFile.write("tokenizer_file=" + tokenizer_file + "\n")
    configFile.write("label_encoder_file=" + label_encoder_file + "\n")
    configFile.close()

    files = []
    files.append(configfile)

    # serialize weights to HDF5
    model.save(model_file)
    files.append(model_file)

    # saving tokenizer
    with open(tokenizer_file, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    files.append(tokenizer_file)

    # saving label_encoder
    with open(label_encoder_file, 'wb') as handle:
        pickle.dump(label_encoder, handle, protocol=pickle.HIGHEST_PROTOCOL)
    files.append(label_encoder_file)


def file_exist(w2v_checkpoint):
    if os.path.exists(w2v_checkpoint):
        return True
    else:
        return False

train_file = "task_informative_text_img_agreed_lab_train.tsv"
dev_file = "task_informative_text_img_agreed_lab_dev.tsv"
test_file = "task_informative_text_img_agreed_lab_test.tsv"

MAX_SEQUENCE_LENGTH = 25

######## Data input ########                    
delim = "\t"
train_x, train_y, train_le, train_labels, word_index, tokenizer = read_train_data(train_file,20000,MAX_SEQUENCE_LENGTH,delim)
dev_x, dev_y, dev_le, dev_labels, _ = read_dev_data(dev_file, tokenizer,MAX_SEQUENCE_LENGTH,delim,train_le)
test_x, test_y, test_le, test_labels, _ = read_dev_data(test_file, tokenizer,MAX_SEQUENCE_LENGTH,delim,train_le)
print("Train: " + str(len(train_x)))

y_true = np.argmax(train_y, axis=1)
y_true = train_le.inverse_transform(y_true)
nb_classes = len(set(y_true.tolist()))
print ("Number of classes: " + str(nb_classes))

######## Word-Embedding ########
if (file_exist("w2v_checkpoint")):
      emb_matrix = pickle.load(open("w2v_checkpoint", "rb"))
else:
  model_file = "crisisNLP_word2vec_model/crisisNLP_word_vector.txt"
  emb_model = KeyedVectors.load_word2vec_format(model_file, binary=False)
  embedding_matrix = prepare_embedding(word_index, emb_model, 20000,300)
  print("Embedding size: " + str(embedding_matrix.shape))
  emb_matrix = embedding_matrix
  vocab_size, embedding_dim = embedding_matrix.shape
  pickle.dump(emb_matrix, open("w2v_checkpoint", "wb"))


#check cuda is available
######## Text network ########                
inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))
cnn = kimCNN(emb_matrix, word_index, 20000,300,MAX_SEQUENCE_LENGTH, inputs)
callback = callbacks.EarlyStopping(monitor='val_acc', patience=200, verbose=0, mode='max')
tensorboard = TensorBoard(log_dir="./checkpoint_log" + "/{}".format(time()), histogram_freq=0, write_graph=True,
                              write_images=True, embeddings_freq=0, embeddings_layer_names="Embedding layer",
                              embeddings_metadata=None)
csv_logger = CSVLogger("./checkpoint_log/log.txt", append=False, separator='\t')
learning_rate_reduction = ReduceLROnPlateau(monitor='val_acc', patience=20, verbose=1, factor=0.01,min_lr=0.00001)
checkpoint = ModelCheckpoint("models/informativeness_cnn_keras.model", monitor='val_acc', verbose=1, save_best_only=True, mode='max')

R, C = train_x.shape
network = Activation('relu')(cnn)
network = Dropout(0.6)(network)
network = Dense(100)(network)
network = Activation('relu')(network)
network = Dense(50)(network)
network = Activation('relu')(network)
out = Dense(nb_classes, activation='softmax',name='lrec-softmax')(network)
model = Model(inputs=inputs, outputs=out)
print("lr=0.00001, beta_1=0.9, beta_2=0.999, amsgrad=False")
#nadam = keras.optimizers.Nadam(lr=0.002, beta_1=0.9, beta_2=0.999)
adam = tf.keras.optimizers.Adam(lr=0.00001, beta_1=0.9, beta_2=0.999, amsgrad=False)
y_true = np.argmax(train_y, axis=1)
class_weights = compute_class_weight(
                                        class_weight = "balanced",
                                        classes = np.unique(y_true),
                                        y = y_true                                                    
                                    )
class_weights = dict(zip(np.unique(y_true), class_weights)),
class_weights

d_class_weights = dict(enumerate(class_weights))
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])


callbacks_list = [callback, learning_rate_reduction, tensorboard, csv_logger, checkpoint]
print(model.summary())
history = model.fit([train_x], train_y, batch_size=128,  epochs=300, verbose=1,validation_data=([dev_x], dev_y), callbacks=callbacks_list)

######## Save the model ########
print ("saving model...")
model.load_weights("models/informativeness_cnn_keras.model")
model.compile(loss='categorical_crossentropy', optimizer="Adam", metrics=['accuracy'])
print ("Best saved model loaded...")

dir_name = os.path.dirname("models/informativeness_cnn_keras.model")
base_name = os.path.basename(train_file)
base_name = os.path.splitext(base_name)[0]
model_dir = dir_name + "/" + base_name + "_text"
save_model(model, model_dir, "models/informativeness_cnn_keras.model", tokenizer, train_le)

######## Evaluation ########    
dev_prob = model.predict([dev_x], batch_size=128, verbose=1)
test_prob = model.predict([test_x], batch_size=128, verbose=1)

##### Dev
out_label_file_name="labeled/informativeness_labeled_cnn.tsv"
dir_name = os.path.dirname(out_label_file_name)
base_name = os.path.basename(out_label_file_name)
base_name = os.path.splitext(base_name)[0]
dev_out_label_file_name = dir_name + "/" + base_name + "_dev_labels.txt"


AUC, accu, P, R, F1, report = performance_measure_cnn(dev_y, dev_prob, train_le)
result = str("{0:.4f}".format(accu)) + "\t" + str("{0:.2f}".format(P)) + "\t" + str(
        "{0:.4f}".format(R)) + "\t" + str("{0:.2f}".format(F1))+ "\t" + str("{0:.4f}".format(AUC)) + "\n"
print(result)
print (report)

out_file = open("results/informativeness_results_cnn.txt", "w")
out_file.write(dev_file + "\n")
out_file.write(result)
out_file.write(report)

###### Test
test_out_label_file_name = dir_name + "/" + base_name + "_test_labels.txt"
AUC, accu, P, R, F1, report = performance_measure_cnn(test_y, test_prob, train_le)
precision = P * 100
recall = R * 100
f1_score = F1 * 100
result = str("{0:.4f}".format(accu)) + "\t" + str("{0:.4f}".format(P)) + "\t" + str(
    "{0:.2f}".format(R)) + "\t" + str("{0:.4f}".format(F1)) + "\t" + str("{0:.4f}".format(AUC))+ "\n"
print("results-cnn:\t"+base_name+"\t"+result)
print (report)
out_file.write( test_file+ "\n")
out_file.write(result)
out_file.write(report)

conf_mat_str = format_conf_mat(test_y, test_prob, train_le)
out_file.write(conf_mat_str+"\n")
out_file.close()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:53: DeprecationWarning: 'U' mode is deprecated


['pls share weâ€™re capturing wildfire response recovery info', 'california wildfires destroy structures kakenews', 'california wildfires destroy structures kakenews', 'california wildfires destroy structures kakenews', 'photos californias destructive wildfires via', 'photos californias destructive wildfires via', 'photos californias destructive wildfires via', 'californias wildfires worse fall', 'californias wildfires worse fall', 'californias wildfires worse fall', 'playing new friend chai shes california fire evacuee starting us days â\x9dï¸', 'playing new friend chai shes california fire evacuee starting us days â\x9dï¸', 'playing new friend chai shes california fire evacuee starting us days â\x9dï¸', 'playing new friend chai shes california fire evacuee starting us days â\x9dï¸', 'calistoga fire tubbsfire napafire abcnow kronnews fire california napa', 'calistoga fire tubbsfire napafire abcnow kronnews fire california napa', 'deadly california wildfires force thousands evacuate â€

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:137: DeprecationWarning: 'U' mode is deprecated


[[0. 0.]
 [0. 0.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]] float64
1573 <class 'int'>
[0 0 0 ... 0 1 1] <class 'numpy.ndarray'>
[   0    1    2 ... 1570 1571 1572] <class 'numpy.ndarray'>
Found 13882 unique tokens.
Shape of data tensor: (1573, 25)
[[0. 0.]
 [0. 0.]
 [0. 0.]
 ...
 [0. 0.]
 [0. 0.]
 [0. 0.]] float64
1534 <class 'int'>
[1 0 0 ... 0 1 1] <class 'numpy.ndarray'>
[   0    1    2 ... 1531 1532 1533] <class 'numpy.ndarray'>
Found 13882 unique tokens.
Shape of data tensor: (1534, 25)
Train: 9601
Number of classes: 2
Preparing embedding matrix.
(None, 25, 300)
lr=0.00001, beta_1=0.9, beta_2=0.999, amsgrad=False
Model: "model_8"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 25)]         0           []                               
                                                           

/usr/local/lib/python3.7/dist-packages/keras/optimizer_v2/adam.py:105: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(Adam, self).__init__(name, **kwargs)


76/76 [==============================] - 18s 228ms/step - loss: 0.6770 - accuracy: 0.6254 - val_loss: 0.6505 - val_accuracy: 0.6726 - lr: 1.0000e-05
Epoch 2/300
76/76 [==============================] - 17s 219ms/step - loss: 0.6409 - accuracy: 0.6618 - val_loss: 0.6168 - val_accuracy: 0.6732 - lr: 1.0000e-05
Epoch 3/300
76/76 [==============================] - 17s 220ms/step - loss: 0.6177 - accuracy: 0.6631 - val_loss: 0.5972 - val_accuracy: 0.6751 - lr: 1.0000e-05
Epoch 4/300
76/76 [==============================] - 17s 224ms/step - loss: 0.6001 - accuracy: 0.6750 - val_loss: 0.5808 - val_accuracy: 0.6968 - lr: 1.0000e-05
Epoch 5/300
76/76 [==============================] - 18s 242ms/step - loss: 0.5863 - accuracy: 0.6981 - val_loss: 0.5686 - val_accuracy: 0.7165 - lr: 1.0000e-05
Epoch 6/300
76/76 [==============================] - 17s 220ms/step - loss: 0.5747 - accuracy: 0.7174 - val_loss: 0.5589 - val_accuracy: 0.7209 - lr: 1.0000e-05
Epoch 7/300
76/76 [===========================

NotFoundError: ignored